# Structural noise from irreducible residuals in pestpp-ies

When the model is missing something the real system has, the residuals left after fitting are
much bigger than the measurement noise the weights describe. Plain ies treats all of that misfit
as information and keeps bending the parameters to chase it, so the posterior parameters come out
biased and the ensemble too narrow.

`ies_reinflate_structnoise_frac` takes the residuals the model can't remove at a reinflation and
turns them into extra noise on the observation targets, so the bias becomes variance instead of
something the parameters bend to fit. `ies_structnoise_deweight` also lowers each observation's
weight by its irreducible residual variance. See section 9.1.17 of the manual.

This notebook runs one small problem three ways:

1. **baseline** - plain ies with reinflation, no structural noise
2. **standard solver, shared weights** - structural noise and de-weighting, control file weights
3. **multimodal solver, unique weights** - the same, with a weight ensemble and phi factors by
   realization, so each realization starts from its own weights

The true parameters are known, so the parameter bias is measured directly, not just phi.

## The test problem

**Time and forcing.** Monthly steps, a 10-year record and a 2-year forecast period. Recharge
$R(t)$ has a winter peak, a seasonal cycle, wet and dry years, and random month-to-month
variation; the second forecast year is dry (60% of normal). Pumping $Q(t)$ runs in the summer
months from year 3. Both are fixed and known to the model.

**The model.** One linear reservoir draining to a boundary, solved exactly over each month and
started at the steady state of the mean forcing over the record:

$$eq(t) = h_b + \frac{rm_z(t)\,R(t) - q_m\,Q(t)}{C}$$

$$h(t+1) = eq(t) + \big(h(t) - eq(t)\big)\,e^{-C/S}$$

$$\text{outflow} = C\,(h - h_b)$$

Two wells sit at different distances from the drain, so they see the same reservoir with
different response factors ($h_{w} = h_b + a_w (h - h_b)$, with $a_1 = 1.0$ and $a_2 = 0.6$).
The outflow is the third output. That gives two observation groups, heads and flows, so the phi
factors have more than one group to balance.

**The parameters.** $\log_{10} S$ (storage), $\log_{10} C$ (drain conductance), $h_b$ (boundary
head), $\log_{10} q_m$ (pumping multiplier) and $\log_{10} rm_z$, a recharge multiplier for each
of 4 time zones over the record (the forecast uses the last zone). All but $h_b$ are in log
space, so the heads are nonlinear in the parameters. The recharge zones are the model's
parameterization only - the truth has one recharge multiplier. The zones and the boundary head
are what plain ies can bend to soak the structure up.

**The truth.** The same reservoir at a set of true parameters drawn from the prior, plus what the
model lacks:

- a slow regional reservoir - a 36-month memory driven by the recharge anomalies, giving
  multi-year swings of about 1.2 m;
- a neighbour's pumping that starts 55% of the way through the record and draws the heads down
  by up to 1.5 m.

Both are seen in full at the wells and at half strength in the outflow, and both carry on
through the forecast period. The observations are the truth itself - no noise is added to them.
The user's stated noise level (0.25 m for heads, 0.003 for flows) goes only into the noise
realizations ies fits to, and the irreducible residuals are several times larger than it.

**The forecasts.** The model runs two years past the record, and the truth is run over the same
period with its structure carrying on, so every forecast has a true value:

- **(a) `fc_h`** - the head at well 1 at the end of the forecast period, an absolute level;
- **(b) `fc_dh`** - the change in that head when the pumping is doubled for the forecast
  period, from a paired run - a difference forecast;
- **(c) `fc_lowq`** - the lowest outflow during the dry forecast year, an extreme mostly
  controlled by storage and conductance.

## Setup

In [ ]:
import os
import sys
import json
import shutil
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyemu

# the pestpp-ies built in this repo
ostag = {"Windows": "win", "Darwin": "mac"}.get(platform.system(), "linux")
exe = os.path.abspath(os.path.join("..", "bin", ostag,
                                   "pestpp-ies" + (".exe" if ostag == "win" else "")))
if not os.path.exists(exe):
    exe = shutil.which("pestpp-ies")
print("pestpp-ies:", exe)

# ensemble size - used for the prior, the noise realizations, the phi factor table and
# ies_num_reals in every case
num_reals = 200

t_d = "structnoise_template"
if os.path.exists(t_d):
    shutil.rmtree(t_d)
os.makedirs(t_d)

# plot colors
c_obs, c_prior, c_post = "#c53030", "#a0aec0", "#2b6cb0"

The model is a python module, run through pestpp's external run manager (`pestpp-ies case.pst
/e`): pestpp writes every run of a batch to `case.rns` and calls the model command once, and
the module fills in all the outputs in one pass. No model binary is needed.

In [ ]:
%%writefile structnoise_template/resmodel.py
"""the reservoir model for the structural noise demo.

one linear reservoir draining to a boundary, solved exactly over each monthly step.  two wells
see the same reservoir with different response factors; the outflow to the boundary is the third
output.  the truth is the same reservoir plus things the model doesnt have: a slow regional
reservoir and a neighbour's pumping that starts partway through the record.
"""
import os
import json
import numpy as np


def default_config():
    return {
        "n_rec": 120,            # monthly record, 10 years
        "n_fc": 24,              # forecast period, 2 years
        "n_zones": 4,            # recharge multiplier zones in time (the model's parameterization)
        "resp": [1.0, 0.6],      # response factors of the two wells
        "forcing_seed": 11,
        "r_base": 0.1, "r_amp": 0.04, "r_noise": 0.02,
        "r_year_sd": 0.25,       # year-to-year wet/dry variation, as a fraction
        "q_rate": 0.03,          # summer pumping from year 3
        "dry_frac": 0.6,         # recharge in the second (dry) forecast year, as a fraction
        "struct_flow": 0.5,      # the drain sees only this fraction of the structural head change
        "slow_amp": 1.2, "slow_memory": 36.0,
        "pump_amp": 1.5, "pump_start_frac": 0.55, "pump_tau": 8.0,
        "prior_mean": [np.log10(0.1), np.log10(0.02), 10.0, 0.0],
        "prior_sd": [0.25, 0.25, 1.5, 0.15],
        "rm_sd": 0.1,
        "head_sd": 0.05, "flow_sd": 0.01,
        "truth_seed": 23,
    }


def true_pars(cfg):
    """the truth: a draw from the prior, but with one recharge multiplier for the whole record -
    the zones are only the model's parameterization"""
    mu, sd = prior_mean_sd(cfg)
    z = np.random.default_rng(cfg["truth_seed"]).standard_normal(5)
    return np.concatenate([mu[:4] + sd[:4] * z[:4], np.full(cfg["n_zones"], sd[4] * z[4])])


def par_names(cfg):
    return ["lg_s", "lg_c", "hb", "lg_qm"] + ["lg_rm{0}".format(z) for z in range(cfg["n_zones"])]


def prior_mean_sd(cfg):
    mu = list(cfg["prior_mean"]) + [0.0] * cfg["n_zones"]
    sd = list(cfg["prior_sd"]) + [cfg["rm_sd"]] * cfg["n_zones"]
    return np.array(mu), np.array(sd)


def series_names(cfg):
    n = cfg["n_rec"] + cfg["n_fc"]
    return {k: ["{0}_{1:03d}".format(k, t) for t in range(n)] for k in ("h1", "h2", "q")}


def obs_names(cfg):
    s = series_names(cfg)
    return s["h1"] + s["h2"] + s["q"] + ["fc_h", "fc_dh", "fc_lowq"]


def forcing(cfg):
    n = cfg["n_rec"] + cfg["n_fc"]
    rng = np.random.default_rng(cfg["forcing_seed"])
    t = np.arange(n)
    mo = t % 12
    r = cfg["r_base"] + cfg["r_amp"] * np.cos(2 * np.pi * mo / 12) + cfg["r_noise"] * rng.standard_normal(n)
    r *= np.maximum(0.2, 1.0 + cfg["r_year_sd"] * rng.standard_normal(n // 12 + 1))[t // 12]
    r = np.maximum(0.0, r)
    r[cfg["n_rec"] + 12:] *= cfg["dry_frac"]
    q = np.where((t >= 24) & (mo >= 5) & (mo <= 8), cfg["q_rate"], 0.0)
    return r, q


def structure(cfg, r):
    """what the truth has that the model doesnt, as a head change in the reservoir"""
    n = len(r)
    nrec = cfg["n_rec"]
    mean_r = r[:nrec].mean()
    slow = np.zeros(n)
    s = 0.0
    for t in range(n):
        s = s * np.exp(-1.0 / cfg["slow_memory"]) + (r[t] - mean_r)
        slow[t] = s
    slow -= slow[:nrec].mean()
    slow *= cfg["slow_amp"] / slow[:nrec].std()
    t = np.arange(n)
    t0 = int(round(cfg["pump_start_frac"] * nrec))
    pump = np.where(t >= t0, cfg["pump_amp"] * (1.0 - np.exp(-(t - t0) / cfg["pump_tau"])), 0.0)
    return slow - pump


def reservoir(p, cfg, r, q, struct=None):
    s, c, hb, qm = 10.0 ** p[0], 10.0 ** p[1], p[2], 10.0 ** p[3]
    nz, nrec = cfg["n_zones"], cfg["n_rec"]
    n = len(r)
    zone = np.minimum(nz - 1, (np.arange(n) * nz) // nrec)
    rm = 10.0 ** np.asarray(p[4:])[zone]
    f = rm * r - qm * q
    decay = np.exp(-c / s)
    h = np.zeros(n)
    hh = hb + f[:nrec].mean() / c
    for t in range(n):
        eq = hb + f[t] / c
        hh = eq + (hh - eq) * decay
        h[t] = hh
    heads = [hb + a * (h - hb) for a in cfg["resp"]]
    flow = c * (h - hb)
    if struct is not None:
        # the structure is seen at the wells in full and only partly at the drain
        heads = [hd + struct for hd in heads]
        flow = flow + c * cfg["struct_flow"] * struct
    return heads[0], heads[1], flow


def outputs(p, cfg, truth=False):
    """every output in obs_names order.  truth=True adds the structural error"""
    r, q = forcing(cfg)
    st = structure(cfg, r) if truth else None
    h1, h2, fl = reservoir(p, cfg, r, q, st)
    # paired run for the difference forecast: pumping doubled in the forecast period
    q2 = q.copy()
    q2[cfg["n_rec"]:] *= 2.0
    h1b, _, _ = reservoir(p, cfg, r, q2, st)
    fc = [h1[-1], h1b[-1] - h1[-1], fl[cfg["n_rec"] + 12:].min()]
    return np.concatenate([h1, h2, fl, fc])


def run_forward(ws="."):
    with open(os.path.join(ws, "resmodel_config.json")) as f:
        cfg = json.load(f)
    vals = {}
    with open(os.path.join(ws, "pars.dat")) as f:
        for line in f:
            if line.strip():
                n, v = line.split()
                vals[n] = float(v)
    y = outputs([vals[n] for n in par_names(cfg)], cfg)
    with open(os.path.join(ws, "outputs.dat"), "w") as f:
        for n, v in zip(obs_names(cfg), y):
            f.write("{0} {1:20.12E}\n".format(n, v))


def run_forward_rns(ws=".", case="case"):
    """the model command for the external run manager (pestpp-ies case.pst /e).  pestpp writes
    every run of the batch to case.rns and calls this once"""
    from pyemu.utils.helpers import RunStor
    with open(os.path.join(ws, "resmodel_config.json")) as f:
        cfg = json.load(f)
    fname = os.path.join(ws, case + ".rns")
    _, pnames, onames = RunStor.file_info(fname)
    rs = RunStor(fname)
    df = rs.get_data()
    lower = {n.lower(): n for n in pnames}
    pcols = [lower[n] for n in par_names(cfg)]
    lower = {n.lower(): n for n in onames}
    ocols = [lower[n] for n in obs_names(cfg)]
    pv = df.loc[:, pcols].values.astype(float)
    df.loc[:, ocols] = np.array([outputs(p, cfg) for p in pv])
    df.loc[:, "run_status"] = 1
    rs.update(df)

In [ ]:
sys.path.insert(0, t_d)
import resmodel

cfg = resmodel.default_config()
with open(os.path.join(t_d, "resmodel_config.json"), "w") as f:
    json.dump(cfg, f, indent=1)

nrec, nfc = cfg["n_rec"], cfg["n_fc"]
n = nrec + nfc
years = (np.arange(n) + 1) / 12.0
snames = resmodel.series_names(cfg)
pnames = resmodel.par_names(cfg)
ptrue = resmodel.true_pars(cfg)
y_truth = resmodel.outputs(ptrue, cfg, truth=True)
y_reach = resmodel.outputs(ptrue, cfg)          # the model at the true parameters
onames = resmodel.obs_names(cfg)
truth = pd.Series(y_truth, index=onames)
reach = pd.Series(y_reach, index=onames)
print(pd.Series(ptrue, index=pnames).round(3).to_string())

### The truth, the model at the true parameters, and the difference

The difference is the structural error - what the residual correction is meant to stop the
parameters chasing. The dashed line marks the end of the record.

In [ ]:
labels = {"h1": "head, well 1 (m)", "h2": "head, well 2 (m)", "q": "outflow"}
fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex=True)
for i, k in enumerate(["h1", "h2", "q"]):
    ax = axes[i, 0]
    ax.plot(years, truth[snames[k]].values, "k--", lw=1.5, label="truth")
    ax.plot(years, reach[snames[k]].values, color=c_post, lw=1.2, label="model at true parameters")
    ax.set_ylabel(labels[k])
    ax = axes[i, 1]
    ax.plot(years, truth[snames[k]].values - reach[snames[k]].values, "k", lw=1.2)
    ax.axhline(0, color="0.6", lw=0.8)
    ax.set_ylabel("structural error")
    for ax in axes[i]:
        ax.axvline(nrec / 12, color="0.4", ls=":", lw=1)
axes[0, 0].legend(loc="upper left", fontsize=8)
axes[-1, 0].set_xlabel("years"); axes[-1, 1].set_xlabel("years")
fig.suptitle("the truth has a slow regional reservoir and a neighbour's pumping the model lacks")
fig.tight_layout()

## Designing the measurement noise

The measurement noise here is autocorrelated in time: within each series (well 1, well 2 and the
outflow) the noise at two times $t_i$ and $t_j$ has correlation $\exp(-|t_i - t_j| / a)$, where
$a$ is the range in months. The standard deviations stay at 0.25 m for the heads and 0.003 for
the outflow. A short range is close to the usual independent noise. A very long range - the
default, 1200 months against a 120-month record - makes each realization's noise nearly a
constant offset over the record, much like a datum or calibration error.

Move the slider to see what the noise looks like, then run the cells below it again to use that
range. The noise model is used for the noise realizations that all three cases share
(`ies_obs_en`); the observed values themselves are the truth, with no noise added. The weights
stay at 1 / standard deviation and know nothing about the correlation.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def corr_noise(n_reals, sd, n_t, range_months, rng):
    # exponential correlation in time, scaled to sd
    t = np.arange(n_t)
    c = np.exp(-np.abs(t[:, None] - t[None, :]) / range_months)
    l = np.linalg.cholesky(c + 1.0e-10 * np.eye(n_t))
    return (rng.standard_normal((n_reals, n_t)) @ l.T) * sd

noise_range = widgets.FloatLogSlider(value=36.0, base=10, min=0.0, max=np.log10(6000.0), step=0.05,
                                     description="range (months)", continuous_update=False,
                                     style={"description_width": "initial"},
                                     layout=widgets.Layout(width="600px"))

def preview(range_months):
    e = corr_noise(10, cfg["head_sd"], nrec, range_months, np.random.default_rng(5))
    fig, ax = plt.subplots(figsize=(12, 3))
    for r in e:
        ax.plot(years[:nrec], r, color=c_obs, lw=0.8, alpha=0.6)
    ax.axhline(0, color="0.5", lw=0.8)
    ax.set_title("10 head noise realizations, range {0:.0f} months".format(range_months))
    ax.set_xlabel("years"); ax.set_ylabel("noise (m)")
    plt.show()

display(noise_range, widgets.interactive_output(preview, {"range_months": noise_range}))

## The pest interface

pyemu writes the template and instruction files and the control file. The record-period heads
and flows carry the stated noise level as a `standard_deviation` column, so weights and noise are
kept separate; the weights start at 1 / standard deviation. The noise realizations themselves are
drawn here from the autocorrelated model above and passed in with `ies_obs_en`, so all three cases
start from the same ones. The forecast-period outputs and the three forecasts have zero weight.
The flows carry a `lower_bound` of 0: the noise realizations drawn here are held at it, and
pestpp-ies applies it to the structural draws at each reinflation.

In [ ]:
with open(os.path.join(t_d, "pars.dat.tpl"), "w") as f:
    f.write("ptf ~\n")
    for p in pnames:
        f.write("{0} ~ {0:^20} ~\n".format(p))
with open(os.path.join(t_d, "pars.dat"), "w") as f:
    for p, v in zip(pnames, ptrue):
        f.write("{0} {1}\n".format(p, v))
with open(os.path.join(t_d, "outputs.dat.ins"), "w") as f:
    f.write("pif ~\n")
    for o in onames:
        f.write("l1 w !{0}!\n".format(o))
with open(os.path.join(t_d, "forward_run.py"), "w") as f:
    f.write("import resmodel\nif __name__ == '__main__':\n    resmodel.run_forward()\n")
with open(os.path.join(t_d, "forward_run_rns.py"), "w") as f:
    f.write("import resmodel\nif __name__ == '__main__':\n    resmodel.run_forward_rns(case='case')\n")
resmodel.run_forward(t_d)

pst = pyemu.Pst.from_io_files([os.path.join(t_d, "pars.dat.tpl")], [os.path.join(t_d, "pars.dat")],
                              [os.path.join(t_d, "outputs.dat.ins")], [os.path.join(t_d, "outputs.dat")],
                              pst_path=".")
mu, sd = resmodel.prior_mean_sd(cfg)
par = pst.parameter_data
par.loc[pnames, "partrans"] = "none"
par.loc[pnames, "parval1"] = mu
par.loc[pnames, "parlbnd"] = mu - 4 * sd
par.loc[pnames, "parubnd"] = mu + 4 * sd
par.loc[pnames, "pargp"] = ["storage", "cond", "bhead", "pump"] + ["rch"] * cfg["n_zones"]

rng = np.random.default_rng(1234)
obs = pst.observation_data
obs.loc[:, "obsval"] = truth.loc[obs.obsnme].values
obs.loc[:, "weight"] = 0.0
obs.loc[:, "obgnme"] = "forecast"
rec_h = snames["h1"][:nrec] + snames["h2"][:nrec]
rec_q = snames["q"][:nrec]
# the series and their noise levels, with the noise range from the slider
series = [(snames["h1"][:nrec], cfg["head_sd"]), (snames["h2"][:nrec], cfg["head_sd"]),
          (snames["q"][:nrec], cfg["flow_sd"])]
a = noise_range.value
print("noise range: {0:.0f} months".format(a))
obs.loc[rec_h, "weight"] = 1.0 / cfg["head_sd"]
obs.loc[rec_h, "obgnme"] = "head"
obs.loc[rec_q, "weight"] = 1.0 / cfg["flow_sd"]
obs.loc[rec_q, "obgnme"] = "flow"
obs.loc[:, "lower_bound"] = np.nan
obs.loc[rec_q, "lower_bound"] = 0.0
# the noise comes from standard_deviation, not the weights, so the two stay separate - the
# de-weighting can lower the weights without touching the user's noise level
obs.loc[:, "standard_deviation"] = np.nan
obs.loc[rec_h, "standard_deviation"] = cfg["head_sd"]
obs.loc[rec_q, "standard_deviation"] = cfg["flow_sd"]

pst.model_command = "python forward_run_rns.py"
pst.pestpp_options = {}
pst.pestpp_options["forecasts"] = "fc_h,fc_dh,fc_lowq"
pst.pestpp_options["ies_num_reals"] = num_reals
pst.pestpp_options["ies_obs_en"] = "noise.csv"
pst.pestpp_options["ies_n_iter_reinflate"] = "-1,999"
pst.control_data.noptmax = 4

np.random.seed(2468)    # pyemu draws from numpy's global generator
cov = pyemu.Cov(x=np.diag(sd ** 2), names=pnames)
pe = pyemu.ParameterEnsemble.from_gaussian_draw(pst, cov=cov, num_reals=num_reals)
pe.enforce()
pe.to_csv(os.path.join(t_d, "prior_pe.csv"))

# the noise realizations every case starts from: observed values plus autocorrelated noise,
# flows held at their lower bound of 0
noise = pd.DataFrame([obs.obsval.values] * num_reals, index=pe.index.astype(str),
                     columns=pst.obs_names, dtype=float)
for nm, s_ in series:
    noise.loc[:, nm] += corr_noise(num_reals, s_, nrec, a, rng)
noise.loc[:, rec_q] = noise.loc[:, rec_q].clip(lower=0.0)
noise.to_csv(os.path.join(t_d, "noise.csv"))
pst.write(os.path.join(t_d, "case.pst"), version=2)
print("parameters:", pst.npar_adj, " weighted observations:", pst.nnz_obs,
      " groups:", sorted(pst.nnz_obs_groups))

### Running a case

Each case gets its own copy of the template and its own options, and runs pestpp-ies with the
external run manager.

In [ ]:
def run_case(name, options, extra_files=None):
    m_d = "structnoise_" + name
    if os.path.exists(m_d):
        shutil.rmtree(m_d)
    shutil.copytree(t_d, m_d)
    p = pyemu.Pst(os.path.join(m_d, "case.pst"))
    p.pestpp_options.update(options)
    p.write(os.path.join(m_d, "case.pst"), version=2)
    for fname, df in (extra_files or {}).items():
        df.to_csv(os.path.join(m_d, fname))
    pyemu.os_utils.run("{0} case.pst /e".format(exe), cwd=m_d)
    return m_d

## Case 1: baseline

Plain ies, with `ies_reinflate_structnoise_frac(0.0)` - the default - and one reinflation after
the first iteration.

In [ ]:
d_base = run_case("baseline", {"ies_reinflate_structnoise_frac": 0.0,"ies_n_iter_reinflate":0})

## Case 2: standard solver, shared weights

The standard (glm) solve, no localization, control file weights only.
`ies_reinflate_structnoise_frac(1.0)` captures the residuals at the first reinflation and uses
them at full strength from then on, and `ies_structnoise_deweight(true)` lowers the weights at
the same time. Every realization gets the same adjusted weights.

The reinflation is after one iteration on purpose. Structure the parameters have already
absorbed never shows up in the residuals, so the later the capture, the less there is to
correct. On this problem, capturing after three iterations instead of one leaves the parameter
error barely better than the baseline.

In [ ]:
sn_opts = {"ies_n_iter_reinflate":[-1,999],"ies_reinflate_structnoise_frac": 1.0, "ies_structnoise_deweight": True}
d_shared = run_case("shared", sn_opts)

## Case 3: multimodal solver, unique weights

`ies_multimodal_alpha(0.25)` and `ies_phi_factor_file` as a table by realization with
`ies_phi_factors_by_real(true)`. No weight ensemble is passed in - every realization starts from
the same control file weights, and pestpp-ies balances each realization's weights to its own row
of phi factors when the prior is run, so each realization ends up with its own weights. The phi
factors run on the prior only - with structural noise on they are never applied again at a
reinflation, since that would count the structure twice. Same structural noise settings as
case 2.

In [ ]:
real_names = [str(i) for i in range(num_reals)]
nz_names = pst.nnz_obs_names
w0 = pst.observation_data.loc[nz_names, "weight"].values
# the share of phi each realization should give the heads, from 50% to 80%
head_frac = np.linspace(0.5, 0.8, len(real_names))
phifac = pd.DataFrame({"head": head_frac, "flow": 1.0 - head_frac}, index=real_names)
phifac.index.name = "real"

d_unique = run_case("unique", dict(sn_opts, ies_multimodal_alpha=0.25,
                                   ies_phi_factor_file="phifac.csv",
                                   ies_phi_factors_by_real=True),
                    extra_files={"phifac.csv": phifac})

## Results

In [ ]:
cases = {"baseline": d_base, "shared weights": d_shared, "unique weights": d_unique}

def last_iter(m_d, tag="par"):
    its = [int(f.split(".")[1]) for f in os.listdir(m_d)
           if f.startswith("case.") and f.endswith("." + tag + ".csv") and f.split(".")[1].isdigit()]
    return max(its)

def load(m_d):
    it = last_iter(m_d)
    pe = pd.read_csv(os.path.join(m_d, "case.{0}.par.csv".format(it)), index_col=0)
    oe = pd.read_csv(os.path.join(m_d, "case.{0}.obs.csv".format(it)), index_col=0)
    pe.columns = pe.columns.str.lower(); oe.columns = oe.columns.str.lower()
    return it, pe, oe

res = {k: load(v) for k, v in cases.items()}
pe0 = pd.read_csv(os.path.join(d_base, "case.0.par.csv"), index_col=0)
oe0 = pd.read_csv(os.path.join(d_base, "case.0.obs.csv"), index_col=0)
pe0.columns = pe0.columns.str.lower(); oe0.columns = oe0.columns.str.lower()
for k, (it, pe, oe) in res.items():
    print("{0:>15}: final iteration {1}, {2} realizations".format(k, it, pe.shape[0]))

### The posteriors against the truth

Realization traces for each output, prior in grey, posterior in blue, observations in red, truth
dashed. Right of the dotted line is the forecast period.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True, sharey="row")
cols = [("prior", oe0)] + [(k, v[2]) for k, v in res.items()]
for j, (title, oe) in enumerate(cols):
    for i, k in enumerate(["h1", "h2", "q"]):
        ax = axes[i, j]
        if title == "prior":
            continue
        color = c_prior if title == "prior" else c_post       
        for r in oe.index:
            ax.plot(years, oe.loc[r, snames[k]].values, color=color, lw=0.5, alpha=0.35)
        ax.plot(years[:nrec], obs.loc[snames[k][:nrec], "obsval"].values, ".", color=c_obs, ms=2.5)
        ax.plot(years, truth[snames[k]].values, "k--", lw=1.2)
        ax.axvline(nrec / 12, color="0.4", ls=":", lw=1)
        if j == 0:
            ax.set_ylabel(labels[k])
    axes[0, j].set_title(title)
for ax in axes[-1]:
    ax.set_xlabel("years")
fig.tight_layout()

### Parameter error

Posterior mean minus truth, and $|z|$ = |mean - truth| / posterior std for each parameter. The
"error" row is the root mean square of (mean - truth) / prior std over all parameters.

In [ ]:
ptrue_s = pd.Series(ptrue, index=pnames)
sd_s = pd.Series(sd, index=pnames)
rows = {}
for k, (it, pe, oe) in [("prior", (0, pe0, oe0))] + list(res.items()):
    m, s = pe[pnames].mean(), pe[pnames].std()
    rows[(k, "mean - truth")] = m - ptrue_s
    rows[(k, "|z|")] = (m - ptrue_s).abs() / s
ptab = pd.DataFrame(rows).T
err = {k: np.sqrt((((pe[pnames].mean() - ptrue_s) / sd_s) ** 2).mean())
       for k, (it, pe, oe) in [("prior", (0, pe0, oe0))] + list(res.items())}
print(ptab.round(3).to_string())
print()
print("parameter error, rms of (mean - truth) / prior std:")
print(pd.Series(err).round(3).to_string())

### Mean residuals against the true structural error

The baseline bends its parameters to fit the structure, so its mean residuals are small and
shaped wrong. With the residual correction the parameters stop chasing it, so the mean residuals
of cases 2 and 3 move toward the true structural error (black). On this problem the move is
partial - it shows most around years 4 to 7, where the neighbour's pumping comes in - and a good
part of the structure is still taken up by the boundary head (see `hb` in the table above).

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
case_colors = {"baseline": c_obs, "shared weights": c_post, "unique weights": "#2f855a"}
for i, k in enumerate(["h1", "h2", "q"]):
    ax = axes[i]
    nm = snames[k][:nrec]
    ax.plot(years[:nrec], truth[nm].values - reach[nm].values, "k", lw=2, label="true structural error")
    for c, (it, pe, oe) in res.items():
        ax.plot(years[:nrec], obs.loc[nm, "obsval"].values - oe[nm].mean().values,
                color=case_colors[c], lw=1.8, label=c)
        ylim = ax.get_ylim()
        for real in oe.index:
            ax.plot(years[:nrec], obs.loc[nm, "obsval"].values - oe.loc[real,nm].values,
                color=case_colors[c], lw=0.3,alpha=0.75,zorder=0)
        ax.set_ylim(ylim)
    ax.axhline(0, color="0.6", lw=0.8)
    ax.set_ylabel("mean residual, " + labels[k])
axes[0].legend(fontsize=8, ncol=4)
axes[-1].set_xlabel("years")
fig.tight_layout()

### The noise realizations before and after the structural draws

`case.obs+noise.csv` is the user's noise; `case.N.obs+noise.csv` is what pestpp-ies rebuilt at the
reinflation (iteration N). The structural draws are combinations of the residuals with zero-mean
coefficients, so the targets spread symmetrically about the observed values.

In [ ]:
# the reinflation iteration - the one that wrote a case.N.obs+noise.csv
it_r = [int(f.split(".")[1]) for f in os.listdir(d_shared)
        if f.endswith(".obs+noise.csv") and f.split(".")[1].isdigit()][0]
print("reinflation at iteration", it_r)
nb = pd.read_csv(os.path.join(d_shared, "case.obs+noise.csv"), index_col=0)
na = pd.read_csv(os.path.join(d_shared, "case.{0}.obs+noise.csv".format(it_r)), index_col=0)
nb.columns = nb.columns.str.lower(); na.columns = na.columns.str.lower()
fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex=True, sharey="row")
for i, k in enumerate(["h1", "q"]):
    nm = snames[k][:nrec]
    for j, (title, ne) in enumerate([("user noise", nb), ("with structural noise", na)]):
        ax = axes[i, j]
        for r in ne.index:
            ax.plot(years[:nrec], ne.loc[r, nm].values, color=c_obs, lw=0.4, alpha=0.3)
        ax.plot(years[:nrec], obs.loc[nm, "obsval"].values, "k", lw=1)
        ax.set_title(title + ", " + labels[k])
fig.tight_layout()

with open(os.path.join(d_shared, "case.rec")) as f:
    rec = f.read()
print([l.strip() for l in rec.split("\n") if "structural noise" in l or "clamped" in l][:12])

### The weights

`case.N.adjusted.weights.csv` holds the weight ensemble each reinflation used. In case 2 every
realization has the same adjusted weights; in case 3 each row was adjusted from its own original
(after the phi factors) and the rows stay distinct. The ratio w'/w is never above 1.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, (c, m_d) in zip(axes, [("shared weights", d_shared), ("unique weights", d_unique)]):
    wa = pd.read_csv(os.path.join(m_d, "case.{0}.adjusted.weights.csv".format(it_r)), index_col=0)
    wa.columns = wa.columns.str.lower()
    if c == "shared weights":
        wb = pd.DataFrame([w0] * wa.shape[0], index=wa.index, columns=nz_names)
    else:
        # the weights each realization started from: the weight ensemble after the phi factors
        wb = pd.read_csv(os.path.join(m_d, "case.adjusted.weights.csv"), index_col=0)
        wb.columns = wb.columns.str.lower()
        wb = wb.loc[wa.index]
    ratio = wa[nz_names] / wb[nz_names]
    for r in ratio.index:
        ax.plot(ratio.loc[r].values, color=c_post, lw=0.5, alpha=0.4)
    ax.set_title("{0}: w'/w, max {1:.3f}, {2} distinct rows".format(
        c, ratio.values.max(), wa[nz_names].round(10).drop_duplicates().shape[0]))
    ax.set_xlabel("observation (heads well 1, heads well 2, flows)")
axes[0].set_ylabel("w' / w")
fig.tight_layout()
print([l.strip() for l in open(os.path.join(d_shared, "case.rec")).read().split("\n")
       if "de-weighting" in l][:3])

### Phi by iteration, actual and measured

All three cases start from the same prior: the same parameter realizations, the same noise
realizations and the same measured phi at iteration 0. In case 3 the phi factors rebalance each
realization's weights on the prior, keeping that realization's measured phi the same, so its
measured prior phi still matches the other two. Its actual prior phi comes out slightly different,
because the rebalanced weights are applied to residuals against the observed values rather than
the noisy targets.

Measured phi jumps at the reinflation where the structural noise comes in, because the noisy
targets now carry the structural draws as well as the measurement noise. Past that point, the
actual phi (against the observed values) is the one to watch.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey="row")
for j, (c, m_d) in enumerate(cases.items()):
    for i, kind in enumerate(["actual", "meas"]):
        df = pd.read_csv(os.path.join(m_d, "case.phi.{0}.csv".format(kind)), index_col=0)
        rcols = [x for x in df.columns if x not in ["total_runs", "mean", "standard_deviation", "min", "max"]]
        ax = axes[i, j]
        for r in rcols:
            ax.plot(df.index, df[r].values, color=c_post, lw=0.5, alpha=0.3)
        ax.plot(df.index, df["mean"].values, color="k", lw=2)
        ax.set_yscale("log")
        ax.set_title("{0}: {1} phi".format(c, kind))
        ax.set_xlabel("iteration")
        ax.hlines([pst.nnz_obs],*ax.get_xlim(),"r",ls="--")
fig.tight_layout()

prior_phi = pd.DataFrame({c: {kind: pd.read_csv(os.path.join(m_d, "case.phi.{0}.csv".format(kind)),
                                                index_col=0).loc[0, "mean"]
                              for kind in ["meas", "actual"]}
                          for c, m_d in cases.items()}).T
print("mean prior phi (iteration 0):")
print(prior_phi.round(1).to_string())

### Forecasts

For each forecast, the prior and the three posteriors as realization values against the true
value (dashed). The residual correction moves the forecasts by correcting the parameters, so
where it helps is where bent parameters extrapolate badly.

On this problem:

- **(b) the difference forecast** gains the most - the bias is about halved. The structure is
  the same in both runs of the pair, so it cancels, and what is left depends on the parameters.
- **(a) the absolute level** does not gain. The structural error carries on into the forecast
  period (about 2 m by the end) and nothing adds it to the forecast - that is the
  forecast-discrepancy limit, and it is left showing here rather than hidden.
- **(c) the low-flow extreme** does not gain either. It is mostly controlled by storage and
  conductance, and the storage error is not corrected here (see `lg_s` in the table above).

In [ ]:
fcs = ["fc_h", "fc_dh", "fc_lowq"]
ens = [("prior", oe0)] + [(k, v[2]) for k, v in res.items()]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
rng_j = np.random.default_rng(0)
for ax, fc in zip(axes, fcs):
    for j, (c, oe) in enumerate(ens):
        v = oe[fc].values
        ax.plot(j + 0.15 * rng_j.standard_normal(len(v)), v, "o", ms=3, alpha=0.5,
                color=c_prior if c == "prior" else c_post)
    ax.axhline(truth[fc], color="k", ls="--", lw=1.5)
    ax.set_xticks(range(len(ens)))
    ax.set_xticklabels([c for c, _ in ens], rotation=20)
    ax.set_title(fc)
fig.tight_layout()

rows = []
for c, oe in ens:
    for fc in fcs:
        v = oe[fc].values
        lo, hi = np.percentile(v, [5, 95])
        rows.append({"case": c, "forecast": fc, "bias": v.mean() - truth[fc], "spread": v.std(),
                     "actual error": abs(v.mean() - truth[fc]),
                     "truth in 5-95%": bool(lo <= truth[fc] <= hi)})
ftab = pd.DataFrame(rows).set_index(["forecast", "case"]).sort_index()
print(ftab.round(4).to_string())

The forecast spread against the actual forecast error, case by case. Cases 2 and 3 are several
times wider than the baseline, but still narrower than their actual error - in (a) by about the
size of the structural error over the forecast period.

In [ ]:
print(ftab[["spread", "actual error"]].unstack("case").round(4).to_string())

## Checks

What the selftest cannot cover: all three cases start from the same measured prior phi, cases 2
and 3 have lower parameter error than the baseline, every adjusted weight is at most its original, and no realization's noise is below
the user's level.

In [ ]:
# every case starts from the same measured prior phi
assert np.allclose(prior_phi["meas"].values, prior_phi["meas"].values[0], rtol=1e-6), prior_phi

assert err["shared weights"] < err["baseline"], err
assert err["unique weights"] < err["baseline"], err

for c, m_d in [("shared weights", d_shared), ("unique weights", d_unique)]:
    wa = pd.read_csv(os.path.join(m_d, "case.{0}.adjusted.weights.csv".format(it_r)), index_col=0)
    wa.columns = wa.columns.str.lower()
    if c == "shared weights":
        wb = pd.DataFrame([w0] * wa.shape[0], index=wa.index, columns=nz_names)
    else:
        wb = pd.read_csv(os.path.join(m_d, "case.adjusted.weights.csv"), index_col=0)
        wb.columns = wb.columns.str.lower()
        wb = wb.loc[wa.index]
    assert (wa[nz_names].values <= wb[nz_names].values * (1 + 1e-6)).all(), c

for c, m_d in [("shared weights", d_shared), ("unique weights", d_unique)]:
    nb = pd.read_csv(os.path.join(m_d, "case.obs+noise.csv"), index_col=0)
    na = pd.read_csv(os.path.join(m_d, "case.{0}.obs+noise.csv".format(it_r)), index_col=0)
    nb.columns = nb.columns.str.lower(); na.columns = na.columns.str.lower()
    ov = obs.loc[nz_names, "obsval"].values
    sb = (nb[nz_names] - ov).std().values
    sa = (na[nz_names] - ov).std().values
    assert (sa >= 0.9 * sb).all(), (c, (sa / sb).min())
print("all checks passed")

## Limits worth knowing

- Structure the first fit already absorbed into the parameters never shows up in the residuals,
  and compensation that leaves no residual can't be seen.
- Real signal that looks like the bias is discounted along with it.
- The forecasts get the parameter correction, but not their own structural error - see (a).
- With the per-observation (diagonal) de-weighting the ensemble comes out overdispersed, because
  the draws are correlated and the weights are not. The parameter correction holds up; the spread
  is too wide.